# Snowflake Python Connector: installation and queries

Run this notebook **locally in Jupyter or VS Code**, not inside a Snowflake-hosted notebook. Install the connector into the Python environment used by its kernel.

| Requirement | Package |
|---|---|
| Connect and execute SQL | `snowflake-connector-python` |
| Fetch results as pandas DataFrames | `snowflake-connector-python[pandas]` |
| Run a local notebook | JupyterLab and/or a VS Code Jupyter kernel (`ipykernel`) |

No JDBC driver, ODBC driver, Java, AWS Python SDK, or Snowpark package is needed for these connector examples. Snowpark and SQLAlchemy serve other APIs and are optional for other work.

Use an existing database/schema and warehouse. The local Python session does not inherit the Snowsight worksheet context. No database/schema/warehouse is created.

## 1. Prepare a local Python environment

If you already have a working local notebook kernel, skip to section 2. Otherwise, run these in Windows PowerShell from this directory (assuming Python 3.12 is installed):

```powershell
py -3.12 -m venv .venv-snow-python
.\.venv-snow-python\Scripts\python.exe -m pip install --upgrade pip
.\.venv-snow-python\Scripts\python.exe -m pip install jupyterlab ipykernel
.\.venv-snow-python\Scripts\python.exe -m ipykernel install --user --name snowflake-python --display-name "Snowflake Python"
.\.venv-snow-python\Scripts\python.exe -m jupyter lab
```

Open this notebook and select the **Snowflake Python** kernel. In VS Code, select the same kernel/environment. Installing the CLI separately does not install the connector into every notebook kernel.

[Connector installation requirements](https://docs.snowflake.com/en/developer-guide/python-connector/python-connector-install)

## 2. Install the connector in this kernel

The pandas extra supports the optional DataFrame section as well. `%pip` targets the current notebook environment. Restart the kernel after installation if imports still use older packages.

In [ ]:
%pip install "snowflake-connector-python[pandas]"

In [ ]:
import sys
import snowflake.connector
print("Python:", sys.executable)
print("Connector:", snowflake.connector.__version__)

The base-only command is `%pip install snowflake-connector-python`. Use the pandas extra above if you will run `fetch_pandas_all()`; let its dependency resolver select compatible Arrow dependencies.

[Connector pandas support](https://docs.snowflake.com/en/developer-guide/python-connector/python-connector-pandas)

## 3. Enter existing connection details

In Snowsight, inspect `CURRENT_USER()`, `CURRENT_ROLE()`, `CURRENT_WAREHOUSE()`, `CURRENT_DATABASE()` and `CURRENT_SCHEMA()`. Enter those existing values below. Use the account identifier from account details, typically `organization-account`, without `https://` or the Snowsight URL.

These prompts collect connection settings only. They do not derive or create object names.

In [ ]:
account = input("Snowflake account identifier: ").strip()
user = input("Snowflake login name: ").strip()
role = input("Existing role: ").strip()
warehouse = input("Existing warehouse: ").strip()
database = input("Existing database: ").strip()
schema = input("Existing schema: ").strip()

## 4. Connect with the authentication your user supports

Choose `password` for password/MFA login or `sso` only if SAML browser SSO is configured. Password and optional MFA passcode inputs are hidden. An empty passcode allows the configured push flow where supported. Complete the account's required MFA challenge.

A browser login to Snowsight alone does not guarantee `externalbrowser` support. For users restricted to PAT or key-pair authentication, obtain the approved method from your administrator instead of disabling authentication controls.

[Python connection and authentication](https://docs.snowflake.com/en/developer-guide/python-connector/python-connector-connect)

In [ ]:
from getpass import getpass

auth_method = input("Authentication (password or sso): ").strip().lower()
if auth_method == "sso":
    conn = snowflake.connector.connect(
        account=account, user=user, authenticator="externalbrowser",
        role=role, warehouse=warehouse, database=database, schema=schema,
    )
elif auth_method == "password":
    password = getpass("Snowflake password: ")
    passcode = getpass("MFA passcode, or Enter for configured push: ")
    try:
        conn = snowflake.connector.connect(
            account=account, user=user, password=password,
            authenticator="username_password_mfa",
            passcode=passcode or None,
            role=role, warehouse=warehouse, database=database, schema=schema,
        )
    finally:
        del password, passcode
else:
    raise ValueError("Choose password or sso.")
print("Connected. Run the context check next.")

## 5. Verify context and fetch a single row

A context manager closes each cursor. Keep `conn` open through the query sections, then close it in section 10.

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE(),
               CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_VERSION()
    """)
    print(cur.fetchone())

## 6. Query orders without creating a table

This small inline dataset makes the notebook usable even if the S3 exercise is not finished. `fetchall()` is appropriate for this bounded result. For large results, fetch in batches rather than loading every row into memory.

[Executing and fetching results](https://docs.snowflake.com/en/developer-guide/python-connector/python-connector-example)

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT COLUMN1::NUMBER AS ORDER_ID,
               COLUMN2::VARCHAR AS STATUS,
               COLUMN3::NUMBER(12,2) AS ORDER_TOTAL
        FROM VALUES (1001, 'NEW', 120.50),
                    (1002, 'SHIPPED', 250.00),
                    (1003, 'NEW', 75.25)
        ORDER BY ORDER_ID
    """)
    print([column[0] for column in cur.description])
    for row in cur.fetchall():
        print(row)
    print("Query ID:", cur.sfqid)

## 7. Bind a filter value safely

Pass data values as parameters, using `%s` with the connector's default parameter style. Do not build SQL by concatenating user input. Bound values are distinct from SQL identifiers such as table names.

In [ ]:
wanted_status = "NEW"
with conn.cursor() as cur:
    cur.execute("""
        SELECT COLUMN1::NUMBER AS ORDER_ID,
               COLUMN2::VARCHAR AS STATUS,
               COLUMN3::NUMBER(12,2) AS ORDER_TOTAL
        FROM VALUES (1001, 'NEW', 120.50),
                    (1002, 'SHIPPED', 250.00),
                    (1003, 'NEW', 75.25)
        WHERE COLUMN2 = %s
        ORDER BY ORDER_ID
    """, (wanted_status,))
    filtered_orders = cur.fetchall()
filtered_orders

Expected: orders 1001 and 1003. To query an existing `ORDERS` table instead, run the optional cell below after completing the S3 load.

In [ ]:
# Optional: requires ORDERS in the selected schema.
with conn.cursor() as cur:
    cur.execute("""
        SELECT ORDER_ID, STATUS, ORDER_TOTAL
        FROM ORDERS
        WHERE STATUS = %s
        ORDER BY ORDER_ID
        LIMIT 100
    """, (wanted_status,))
    for row in cur.fetchall():
        print(row)

## 8. Fetch a DataFrame and export locally

This standalone query works without a saved table. The export is a local CSV file in the notebook's current working directory; it does not upload to Snowflake or S3.

[DataFrame fetching](https://docs.snowflake.com/en/developer-guide/python-connector/python-connector-pandas)

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT COLUMN1::NUMBER AS ORDER_ID,
               COLUMN2::VARCHAR AS STATUS,
               COLUMN3::NUMBER(12,2) AS ORDER_TOTAL
        FROM VALUES (1001, 'NEW', 120.50), (1002, 'SHIPPED', 250.00)
        ORDER BY ORDER_ID
    """)
    orders_df = cur.fetch_pandas_all()
display(orders_df)
orders_df.to_csv("orders_python_export.csv", index=False)

## 9. Optional: query the S3 stage and external table

Run only after creating the matching objects in the S3 notebook. These calls use Snowflake SQL and the stage's stored AWS credentials. No `boto3` installation or local AWS key is necessary.

A failed S3 read can mean sandbox credentials expired even if the Python-to-Snowflake connection is healthy.

In [ ]:
# Optional: requires the S3 lab stage and its refreshed directory metadata.
with conn.cursor() as cur:
    cur.execute("LIST @ORDERS_S3_STAGE")
    for row in cur.fetchmany(10):
        print(row)
    cur.execute("""
        SELECT RELATIVE_PATH, SIZE
        FROM DIRECTORY(@ORDERS_S3_STAGE)
        ORDER BY RELATIVE_PATH
    """)
    print(cur.fetchall())

In [ ]:
# Optional: requires the external table from the S3 notebook.
with conn.cursor() as cur:
    cur.execute("""
        SELECT ORDER_ID, STATUS, ORDER_TOTAL
        FROM ExternalOrderTable
        ORDER BY ORDER_ID
        LIMIT 20
    """)
    print(cur.fetchall())

## 10. Close the connection

Run this cell even if an optional query fails. For a standalone script, place the connection's work in `try/finally` and call `conn.close()` in `finally`. Reconnect before repeating queries after closing.

In [ ]:
conn.close()
print("Connection closed.")

## 11. Troubleshooting

| Problem | Action |
|---|---|
| `ModuleNotFoundError` | Check `sys.executable`, install with `%pip`, then restart that kernel |
| No suitable package version | Check the connector's supported Python versions and update pip |
| Authentication/MFA failure | Confirm your user's permitted method; renew the passcode or complete the challenge |
| `externalbrowser` fails | Confirm SAML SSO setup and browser access from the local machine |
| Table not found / not authorized | Check database, schema and role in section 5; skip optional S3 cells until objects exist |
| Warehouse missing or inaccessible | Select an existing permitted warehouse in the connection inputs |
| pandas/Arrow import failure | Install the connector's pandas extra in this kernel; avoid mismatched Arrow pins |
| Network or certificate error | Use approved proxy/network configuration; do not turn off TLS verification |

The connector can execute COPY, INSERT and other SQL too, subject to privileges. This notebook uses read-only examples to keep the existing orders data intact. Its Python syntax and notebook structure are validated; live connectivity requires your account.